In [1]:
import sys
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor 
from wsheet20 import Wsheet20, Sheet, ColeCole, Source
!chmod +x /home/vanza334/projects/def-sgkang09/vanza334/python_wrapper/a.out

In [2]:
os.chdir('/home/python_wrapper') # Example of the path

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor 
from wsheet20 import Wsheet20, Sheet, ColeCole, Source

def create_run_for_frequency(freq_hz, output_dir):
    # Create a simulation run for a specific frequency and save output in output directory
    run = Wsheet20(
        sheets=[
            Sheet(
                na=20, nb=20,  # Number of cells along strike and dip directions for the sheet
                center=(500., 0., -200.),  # Sheet center coordinates (x=500m, y=0m, depth of the center of the sheet=200m)
                strike_length=200., dip_length=200.,  # Physical size of the sheet (meters)
                strike=90., dip=90., thickness=0.1,  # Orientation angles (degrees) and thickness (meters)
                resistivity=ColeCole(low=1e-5, high=1e-5, tau=0., alpha=0.5),  # Electrical resistivity model for the sheet
                permittivity=ColeCole(low=2., high=2., tau=0., alpha=0.5),  # Electrical permittivity model for the sheet
            )
        ],
        host_resistivity= ColeCole(low=10., high=10., tau=0., alpha=0.5),  # Background resistivity of host earth
        host_permittivity= ColeCole(low=10., high=10., tau=0., alpha=0.5),  # Background permittivity of host earth
        frequency= freq_hz,  # Frequency of the simulation (Hz)
        source= Source(type="magnetic_dipole", direction="z", angle=0.),  # Electromagnetic source and direction 
        n_tx=1,  # Number of transmitters
        tx_start=(0., 0., 0.),  # Starting location of transmitter
        tx_increment=(0., 0., 0.),  # Spacing between transmitter along x, y, z axes
        n_rx=50,  # Number of receivers measuring the response
        rx_start=(0., 0., 0.75),  # Receiver start location
        rx_increment=(5., 0., 0.),  # Spacing between receivers along x, y, z axes
        green_mode=0,  # Parameter controlling Green's function calculation mode
        output_scatter=True,  # Save scattered field output
        output_efield=True,  # Save electric field output
        output_hfield=True,  # Save magnetic field output
        normalize=False,  # Do not normalize output fields
    )
    
    # Example path to the Fortran executable that runs the simulation
    run.executable = Path('home/python_wrapper') / "a.out"
    
    # Create a directory named for this frequency within the main output directory
    freq_dir = output_dir / f"freq_{int(freq_hz)}"
    freq_dir.mkdir(exist_ok=True)  # Create directory if it doesn't already exist
    run.work_dir = freq_dir  # Set the working directory for this simulation run
    
    print(f"Starting run for frequency: {freq_hz} Hz")  # Simulation is starting
    run.run()  # Execute the simulation
    print(f"Finished run for frequency: {freq_hz} Hz")  # Simulation is finished
    
    # Return frequency and output directory for tracking
    return freq_hz, freq_dir

# List of frequencies (in Hz) to simulate
frequencies = [5e3, 10e3, 20e3]  

# Example of the base output directory to store all simulation results
output_dir = Path('/home/python_wrapper/output')
output_dir.mkdir(exist_ok=True)  # Create the output directory if it doesn't exist

# Controls parallel execution of simulations to speed up processing
with ThreadPoolExecutor(max_workers=3) as executor:  # # Limits number of concurrent simulations
    # Submits jobs for different frequencies
    futures = [executor.submit(create_run_for_frequency, f, output_dir) for f in frequencies]
    
    # As each simulation finishes, retrieve its result and print a completion message
    for future in futures:
        freq, outdir = future.result()
        print(f"Completed frequency {freq} Hz, results in {outdir}")

Starting run for frequency: 5000.0 Hz
Starting run for frequency: 10000.0 Hz
Starting run for frequency: 20000.0 Hz
[wsheet20] Parameter file written → /home/vanza334/projects/def-sgkang09/vanza334/python_wrapper/output/freq_20000/wsheet20.par
[wsheet20] Parameter file written → /home/vanza334/projects/def-sgkang09/vanza334/python_wrapper/output/freq_5000/wsheet20.par
[wsheet20] Parameter file written → /home/vanza334/projects/def-sgkang09/vanza334/python_wrapper/output/freq_10000/wsheet20.par
[wsheet20] Running: /home/vanza334/projects/def-sgkang09/vanza334/python_wrapper/a.out
[wsheet20] Running: /home/vanza334/projects/def-sgkang09/vanza334/python_wrapper/a.out
[wsheet20] Running: /home/vanza334/projects/def-sgkang09/vanza334/python_wrapper/a.out
 >>> GREEN FUNCTION $ POTENTIAL EVALUATION
      For Sheet No. S           1
 >>> GREEN FUNCTION $ POTENTIAL EVALUATION
      For Sheet No. S           1
 >>> GREEN FUNCTION $ POTENTIAL EVALUATION
      For Sheet No. S           1
 >>> LU D